In [1]:
from env import config
import requests
import json

In [2]:
flightDate = "2022-04-21"
sql_query = f"""
SELECT * FROM mindsdb.flight_price_predictor AS m
JOIN ai_travel_agent.flight_prices AS t
WHERE t.flightDate > "{flightDate}"
AND t.startingAirport = "SFO"
AND t.isNonStop = 1
AND t.destinationAirport = "BOS";
"""

In [3]:
def mindsdb_query(sql_query):
    url = 'http://127.0.0.1:47334/api/sql/query'
    headers = {"Content-Type": "application/json"}
    return requests.post(url, json={'query':sql_query}, headers=headers)

In [4]:
def predict_query(flightDate = "2022-04-21", startingAirport="SFO", isNonStop=1, destinationAirport="BOS", raw_request=False):
    sql_query = f"""
        SELECT m.flightDate, m.segmentsAirlineName, m.isNonStop, m.totalFare, m.totalFare_confidence FROM mindsdb.flight_price_predictor AS m
        JOIN ai_travel_agent.flight_prices AS t
        WHERE t.flightDate = "{flightDate}"
        AND t.startingAirport = "{startingAirport}"
        AND t.isNonStop = {isNonStop}
        AND t.destinationAirport = "{destinationAirport}";
    """
    response = mindsdb_query(sql_query)
    response.raise_for_status()
    if raw_request:
        return response
    
    data = response.json()
    columns = data.get("column_names")
    dataset = data.get("data")
    web_ready_data = [dict(zip(columns, row)) for row in dataset]
    return web_ready_data

In [5]:
query_output = predict_query(flightDate = "2022-04-21", startingAirport="BOS", isNonStop=1, destinationAirport="SFO", raw_request=False)
query_output

[{'flightDate': '2022-04-22 00:00:00.000000',
  'segmentsAirlineName': 'American Airlines',
  'isNonStop': '1',
  'totalFare': 50.365632744590144,
  'totalFare_confidence': 0.9991},
 {'flightDate': '2022-04-23 00:00:00.000000',
  'segmentsAirlineName': 'American Airlines',
  'isNonStop': '1',
  'totalFare': 34.968828380903425,
  'totalFare_confidence': 0.9991},
 {'flightDate': '2022-04-24 00:00:00.000000',
  'segmentsAirlineName': 'American Airlines',
  'isNonStop': '1',
  'totalFare': 84.03911508363528,
  'totalFare_confidence': 0.9991},
 {'flightDate': '2022-04-25 00:00:00.000000',
  'segmentsAirlineName': 'American Airlines',
  'isNonStop': '1',
  'totalFare': 19.31460446411327,
  'totalFare_confidence': 0.9991},
 {'flightDate': '2022-04-26 00:00:00.000000',
  'segmentsAirlineName': 'American Airlines',
  'isNonStop': '1',
  'totalFare': 3.0482105879128376,
  'totalFare_confidence': 0.9991},
 {'flightDate': '2022-04-27 00:00:00.000000',
  'segmentsAirlineName': 'American Airlines',


In [6]:
pref = dict(flightDate = "2022-04-21", startingAirport="LAX", isNonStop=1, destinationAirport="JFK")
query_output = predict_query(**pref, raw_request=False)
options = query_output
rec_context = json.dumps({"options": options, "preferences": pref})
rec_context

'{"options": [{"flightDate": "2022-04-22 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 19.549687792198995, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-23 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 17.831208453081487, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-24 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 97.47756341967188, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-25 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 13.165486132827429, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-26 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 0.47940162549013354, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-27 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 53.2593379436

In [20]:
rec_sql_query = f"""
SELECT answer
FROM ai_travel_agent
WHERE question = 'What is the best flight option? Respond in JSON format using curly brackets instead of ```json'
AND context = '{rec_context}';
"""
rec_sql_query

'\nSELECT answer\nFROM ai_travel_agent\nWHERE question = \'What is the best flight option? Respond in JSON format using curly brackets instead of ```json\'\nAND context = \'{"options": [{"flightDate": "2022-04-22 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 19.549687792198995, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-23 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 17.831208453081487, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-24 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 97.47756341967188, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-25 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 13.165486132827429, "totalFare_confidence": 0.9991}, {"flightDate": "2022-04-26 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 0.47940162549

In [21]:
rec_prediction_response = mindsdb_query(rec_sql_query)
rec_prediction_response.json()

{'type': 'table',
 'data': [['{"flightDate": "2022-04-26 00:00:00.000000", "segmentsAirlineName": "JetBlue Airways", "isNonStop": "1", "totalFare": 0.47940162549013354, "totalFare_confidence": 0.9991}\n']],
 'column_names': ['answer'],
 'context': {'show_secrets': False, 'db': 'mindsdb'}}

In [24]:
def recommended_flight(
    user_data = {},
    forecast_data = [],
    question = "What is the best flight option? Respond in JSON format using curly brackets instead of ```json",
    raw_request=False, 
    **kwargs):
    
    context = {"options": forecast_data, "preferences": user_data}
    context_data = json.dumps(context)
    
    sql_query = f"""
        SELECT answer
        FROM ai_travel_agent
        WHERE question = '{question}' 
        AND context = '{context_data}';
    """
    
    response = mindsdb_query(sql_query)
    response.raise_for_status()
    if raw_request:
        return response
    
    data = response.json()
    dataset = data.get("data")
    if dataset is None or data is None:
        return []
    
    if isinstance(dataset, list):
        sub_dataset = dataset[0]
        if isinstance(sub_dataset, list):
            return sub_dataset[0]
        return sub_dataset
    
    return dataset


In [27]:
json.loads(recommended_flight(user_data = pref, forecast_data=query_output))

{'flightDate': '2022-04-26 00:00:00.000000',
 'segmentsAirlineName': 'JetBlue Airways',
 'isNonStop': '1',
 'totalFare': 0.47940162549013354,
 'totalFare_confidence': 0.9991}